
# HKH Lake Clusters → Single AOI per Cluster (Bugfix)

**Fix:** Exports **one** GeoJSON per shortlisted cluster (single feature).  
Previously, AOIs were split into parts and the shortlist picked the top 20 **parts** from the largest cluster — yielding multiple files for one cluster. This version aggregates by `cluster_id` and writes a single AOI per cluster.


## Imports

In [1]:

import math
from pathlib import Path

import geopandas as gpd
from shapely.ops import unary_union
from shapely import force_2d
from sklearn.cluster import DBSCAN
import pandas as pd


## Configuration

In [2]:

# === Paths ===
LAKES_VECTOR_PATH = "HKH-PK.zip"      # single file with many polygons (one per lake)
OUT_DIR = "aoi_geojsons_cluster"       # output directory for per-cluster AOIs
SUMMARY_CSV = "aoi_geojsons_cluster/cluster_summary.csv"

# === Clustering & AOI params ===
CLUSTER_RADIUS_M = 5000       # DBSCAN radius (meters) on centroids
AOI_BUFFER_M = 100            # buffer around merged cluster geometry (meters)
MIN_LAKES_PER_CLUSTER = 1     # filter small clusters if desired
TOP_K = 20                    # number of clusters (by lake_count) to export

EXPORT_EPSG = "EPSG:4326"

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("Configured. Edit paths & params above as needed.")


Configured. Edit paths & params above as needed.


## Utilities

In [3]:

def pick_local_utm_epsg(lon: float, lat: float) -> str:
    zone = int(math.floor((lon + 180) / 6) + 1)
    south = lat < 0
    epsg = 32700 + zone if south else 32600 + zone
    return f"EPSG:{epsg}"

def reproject_to_local_metric(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    c = gdf.to_crs(4326).unary_union.centroid
    utm = pick_local_utm_epsg(c.x, c.y)
    return gdf.to_crs(utm), utm

def compute_area_km2_metric_crs(geom) -> float:
    return float(gpd.GeoSeries([geom]).area.iloc[0]) / 1e6

def make_points_gdf_from_centroids(lakes_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    pts = lakes_gdf.geometry.centroid
    return gpd.GeoDataFrame(lakes_gdf.drop(columns="geometry"), geometry=pts, crs=lakes_gdf.crs)

def dbscan_cluster_labels(points_gdf: gpd.GeoDataFrame, eps_m: float) -> pd.Series:
    XY = pd.DataFrame({
        'x': points_gdf.geometry.x.values,
        'y': points_gdf.geometry.y.values
    })
    labels = DBSCAN(eps=eps_m, min_samples=1, metric='euclidean').fit_predict(XY.values)
    return pd.Series(labels, index=points_gdf.index)

def export_geojson(gdf: gpd.GeoDataFrame, path: str):
    gdf.to_crs(4326).to_file(path, driver="GeoJSON")


## Pipeline (one AOI per cluster)

In [4]:

# 1) Load lakes
lakes = gpd.read_file(LAKES_VECTOR_PATH)
assert not lakes.empty, "Lakes layer is empty."
lakes = lakes.explode(ignore_index=True)
lakes = lakes.set_geometry(lakes.geometry.buffer(0))

# 2) Reproject to local metric CRS
lakes_ll = lakes.to_crs(4326)
local_metric_gdf, LOCAL_EPSG = reproject_to_local_metric(lakes_ll)
print(f"Using local metric CRS: {LOCAL_EPSG}")

# 3) Cluster centroids
points = make_points_gdf_from_centroids(local_metric_gdf)
labels = dbscan_cluster_labels(points, eps_m=CLUSTER_RADIUS_M)
local_metric_gdf["cluster_id"] = labels.values

# 4) Build ONE AOI per cluster (merge+buffer), compute area, count lakes
cluster_rows = []
for cid, grp in local_metric_gdf.groupby("cluster_id"):
    lake_count = len(grp)
    if lake_count < MIN_LAKES_PER_CLUSTER:
        continue
    merged = unary_union(grp.geometry.values)
    merged = force_2d(merged)
    aoi_geom = merged.buffer(AOI_BUFFER_M).buffer(0)
    area = compute_area_km2_metric_crs(aoi_geom)
    cluster_rows.append({
        "cluster_id": int(cid),
        "lake_count": int(lake_count),
        "area_km2": float(round(area, 3)),
        "geometry": aoi_geom
    })

clusters_gdf = gpd.GeoDataFrame(cluster_rows, geometry="geometry", crs=local_metric_gdf.crs)

# 5) Shortlist top-K clusters by lake_count (ties resolved by smaller area)
shortlisted = clusters_gdf.sort_values(["lake_count","area_km2"], ascending=[False, True]).head(TOP_K).copy()
shortlisted["shortlist_rank"] = range(1, len(shortlisted)+1)

# 6) Export ONE GeoJSON per cluster
summary = shortlisted[["shortlist_rank","cluster_id","lake_count","area_km2"]].copy()
summary_path = Path(SUMMARY_CSV)
summary.to_csv(summary_path, index=False)

exported_paths = []
for _, row in shortlisted.iterrows():
    cid = row.cluster_id
    rank = row.shortlist_rank
    lk = row.lake_count
    area = row.area_km2
    out_name = f"cluster_{cid:05d}_rank_{rank:02d}_lakes_{lk}_area_{area:.1f}km2.geojson"
    out_path = Path(OUT_DIR) / out_name
    export_geojson(gpd.GeoDataFrame([row], geometry=[row.geometry], crs=local_metric_gdf.crs), str(out_path))
    exported_paths.append(str(out_path))

print("\n=== Shortlist (top-K by lake count) ===")
print(summary.to_string(index=False))
print(f"\nWrote {len(exported_paths)} cluster GeoJSONs to: {OUT_DIR}")
print(f"Summary CSV: {summary_path}")


C:\Users\gg\AppData\Local\Temp\ipykernel_9348\434331413.py:8: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = gdf.to_crs(4326).unary_union.centroid


Using local metric CRS: EPSG:32643

=== Shortlist (top-K by lake count) ===
 shortlist_rank  cluster_id  lake_count  area_km2
              1          40        1031   120.823
              2          85        1026    29.328
              3          84         682    26.292
              4          89         539    16.786
              5           0         468    46.461
              6         176         364    34.037
              7         132         329    10.770
              8         178         294    28.043
              9          88         285     9.606
             10         144         237     6.590
             11          66         202    21.201
             12          15         188     7.307
             13         122         186     7.694
             14         174         171    16.069
             15          20         159     9.378
             16          24         145    24.711
             17           3         127    13.438
             18         